# 08 — Eval (RunPod / GPU)

Runs the full BabyLM 2026 evaluation pipeline against a trained model repo and pushes all results to a separate HF dataset repo you own.

**What this notebook does:**
1. Installs eval dependencies and downloads eval data
2. Runs fast zero-shot eval (`eval_zero_shot_fast.sh`) on every `chck_NM` revision that exists on Hub
3. Runs full zero-shot eval (`eval_zero_shot.sh`) on the final model (main branch)
4. Runs fine-tuning eval (`eval_finetuning.sh`) on the final model
5. Runs Age of Acquisition eval (`eval_aoa.sh`) on the final model
6. Runs GlobalPIQA eval (`eval_zero_shot_global_piqa.sh`) on main + all checkpoints
7. Collates predictions into a submission-ready JSON (`collate_preds.sh`)
8. Pushes the full `results/` directory + submission JSON to `RESULTS_REPO` under a subfolder named after the model

**Can be run on the same pod as notebook 07 or a fresh one.**
All results are pushed to Hub so nothing is lost if the pod is interrupted.

**Compare runs:** use notebook 09 (local) to pull results from multiple repos and plot side-by-side.

In [ ]:
# ── Cell 1: RunPod setup (run once per session) ──
# Uncomment if starting a fresh pod without the eval repo already cloned.

# !git clone https://github.com/babylm-org/babylm-eval.git /workspace/babylm-eval

print('Setup complete.')

In [ ]:
# ── Cell 2: Config ──
import os

HUB_MODEL_REPO = None          # "eligran12/babylm_CL2_gunning_fog" — the repo to evaluate
HF_TOKEN       = os.environ.get("HF_TOKEN")
HF_USER        = "flakoash"

RESULTS_REPO   = f"{HF_USER}/babylm-eval-results"  # your own dataset repo — must exist on HF

BACKEND = "causal"             # GPT-2 is a causal LM
TRACK   = "strict-small"

EVAL_REPO_DIR  = "/workspace/babylm-eval/strict"   # path to the cloned eval repo's strict/ dir
EVAL_DATA_DIR  = f"{EVAL_REPO_DIR}/evaluation_data"
RESULTS_DIR    = f"{EVAL_REPO_DIR}/results"

# Competition revision names for Strict-Small (must match what training pushed).
# chck_1M … chck_9M then chck_10M … chck_100M
FAST_REVISIONS = (
    [f"chck_{i}M"    for i in range(1, 10)] +
    [f"chck_{i*10}M" for i in range(1, 11)]
)

assert HUB_MODEL_REPO, "Set HUB_MODEL_REPO in this cell."
print(f"Model repo   : {HUB_MODEL_REPO}")
print(f"Results repo : {RESULTS_REPO}")
print(f"Backend      : {BACKEND}  |  Track: {TRACK}")
print(f"Revisions    : {FAST_REVISIONS}")

In [ ]:
# ── Cell 3: Install eval requirements + download eval data ──
import subprocess, sys, os
from pathlib import Path

# Redirect HF downloads to the pod volume (/workspace) which has free space.
# The container disk fills up quickly with ~400 MB per checkpoint revision.
HF_CACHE_DIR = Path("/workspace/hf-cache")
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Ensure bash subprocesses (eval_*.sh scripts) use the same python binary as
# this kernel, and write HF downloads to the pod volume.
_py_bin_dir = str(Path(sys.executable).parent)
_env = os.environ.copy()
_env["PATH"]     = _py_bin_dir + ":" + _env.get("PATH", "")
_env["HF_HOME"]  = str(HF_CACHE_DIR)   # huggingface_hub ≥ 0.8
_env["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR / "hub")  # older transformers fallback

def run(cmd, cwd=None, check=True):
    """Run a shell command, stream output, raise on failure."""
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd, shell=True, cwd=cwd, env=_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    return result

# Install eval dependencies
run(f"{sys.executable} -m pip install -r requirements.txt -q", cwd=EVAL_REPO_DIR)

# The pod image ships an older transformers that pip leaves in place when it
# already satisfies a loose version bound. Install the exact pinned version
# without --force-reinstall so torch and other GPU packages are not touched.
run(f"{sys.executable} -m pip install 'transformers==4.51.3' 'huggingface_hub==0.31.0' -q")

# torchvision from the pod image can conflict with the installed torch version and
# is not needed for text-only eval — remove it to avoid the nms operator error.
run(f"{sys.executable} -m pip uninstall torchvision torchaudio -y", check=False)

# Create empty .env so eval_finetuning.sh doesn't fail on 'source ../.env'
env_path = Path(EVAL_REPO_DIR) / ".env"
if not env_path.exists():
    env_path.write_text("")
    print("Created empty .env")

# Download eval data (skips if already present)
fast_dir = Path(EVAL_DATA_DIR) / "fast_eval"
full_dir = Path(EVAL_DATA_DIR) / "full_eval"
if fast_dir.exists() and full_dir.exists():
    print("Eval data already present — skipping download.")
else:
    print("Downloading eval data from Hub...")
    run(f"{sys.executable} scripts/download_evals.py", cwd=EVAL_REPO_DIR)

print(f"\nKernel python  : {sys.executable}")
print(f"HF cache dir   : {HF_CACHE_DIR}")
print("Eval environment ready.")

In [ ]:
# ── Cell 4: Discover which revisions exist on Hub ──
# Only run fast eval for revisions that were actually pushed during training.

from huggingface_hub import list_repo_refs

remote_branches = {
    b.name
    for b in list_repo_refs(HUB_MODEL_REPO, repo_type="model", token=HF_TOKEN).branches
}

available = [r for r in FAST_REVISIONS if r in remote_branches]
missing   = [r for r in FAST_REVISIONS if r not in remote_branches]

print(f"Available revisions ({len(available)}): {available}")
if missing:
    print(f"Missing   revisions ({len(missing)}): {missing}")
    print("  → these will be skipped (scored as 0 by collate_preds)")

In [ ]:
# ── Cell 4b: Repair tokenizer_config on existing HF branches ──
# Our tokenizer saves with tokenizer_class=TokenizersBackend which AutoProcessor
# cannot resolve. This cell patches every chck_NM branch AND main on Hub to use
# PreTrainedTokenizerFast instead. Safe to re-run — skips already-patched branches.

import json, tempfile
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
patched, skipped = [], []

for revision in available + ["main"]:
    try:
        cfg_str = open(hf_hub_download(
            HUB_MODEL_REPO, "tokenizer_config.json",
            revision=revision, repo_type="model", token=HF_TOKEN,
        )).read()
        cfg = json.loads(cfg_str)

        if cfg.get("tokenizer_class") in (None, "PreTrainedTokenizerFast",
                                          "GPT2Tokenizer", "GPT2TokenizerFast"):
            skipped.append(revision)
            continue

        cfg["tokenizer_class"] = "PreTrainedTokenizerFast"
        for k in ("backend", "is_local", "local_files_only"):
            cfg.pop(k, None)

        with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as f:
            json.dump(cfg, f, indent=2)
            tmp = f.name

        api.upload_file(
            path_or_fileobj=tmp,
            path_in_repo="tokenizer_config.json",
            repo_id=HUB_MODEL_REPO,
            repo_type="model",
            revision=revision,
            token=HF_TOKEN,
            commit_message=f"fix tokenizer_class for eval compatibility ({revision})",
        )
        patched.append(revision)
        print(f"  patched {revision}")
    except Exception as e:
        print(f"  {revision}: {e}")

print(f"\nPatched: {patched}")
print(f"Already OK / skipped: {skipped}")

In [ ]:
# ── Cell 5: Fast zero-shot eval — one revision at a time ──
# Runs eval_zero_shot_fast.sh for each available chck_NM revision.
# Clears the HF model cache after each revision so only one ~400 MB model
# is on disk at a time (cache lives on /workspace to avoid filling container disk).

import shutil
from pathlib import Path

fast_eval_dir = "evaluation_data/fast_eval"
skipped, ran, failed = [], [], []
MODEL_NAME = HUB_MODEL_REPO.split("/")[-1]   # eval scripts strip the org prefix

for revision in available:
    # Eval scripts save under just the model name, not the full org/model path.
    result_path = Path(RESULTS_DIR) / MODEL_NAME / revision / "zero_shot"
    if result_path.exists() and any(result_path.iterdir()):
        print(f"  {revision}: results already present — skipping.")
        skipped.append(revision)
        continue

    print(f"\n{'='*60}")
    print(f"Fast zero-shot eval: {revision}")
    print(f"{'='*60}")
    result = run(
        f"bash scripts/eval_zero_shot_fast.sh {HUB_MODEL_REPO} {revision} {BACKEND} {fast_eval_dir}",
        cwd=EVAL_REPO_DIR,
        check=False,
    )
    if result.returncode == 0:
        ran.append(revision)
    else:
        print(f"  ⚠ {revision} failed (exit {result.returncode}) — continuing.")
        failed.append(revision)

    # Clear HF cache after each revision — keep only one model on disk at a time.
    if HF_CACHE_DIR.exists():
        shutil.rmtree(HF_CACHE_DIR)
        HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        print(f"  HF cache cleared after {revision}.")

print(f"\nFast eval done — ran: {len(ran)}  skipped: {len(skipped)}  failed: {len(failed)}")
if failed:
    print(f"  Failed: {failed}")

In [ ]:
# ── Cell 6: Full zero-shot eval — final model (main branch) ──

full_eval_dir = "evaluation_data/full_eval"
MODEL_NAME    = HUB_MODEL_REPO.split("/")[-1]
result_path   = Path(RESULTS_DIR) / MODEL_NAME / "main" / "zero_shot"

if result_path.exists() and any(result_path.iterdir()):
    print("Full zero-shot results already present — skipping.")
else:
    print("Running full zero-shot eval on final model...")
    run(
        f"bash scripts/eval_zero_shot.sh {HUB_MODEL_REPO} {BACKEND} {full_eval_dir}",
        cwd=EVAL_REPO_DIR,
    )
    print("Full zero-shot eval complete.")

In [ ]:
# ── Cell 7: Fine-tuning eval — final model ──
# Runs GLUE tasks (boolq, multirc, rte, wsc, mrpc, qqp, mnli).
# This is the slowest cell — expect 30–90 min on a GPU.

MODEL_NAME           = HUB_MODEL_REPO.split("/")[-1]
finetune_result_path = Path(RESULTS_DIR) / MODEL_NAME / "main" / "finetune"

if finetune_result_path.exists() and any(finetune_result_path.iterdir()):
    print("Fine-tuning results already present — skipping.")
else:
    print("Running fine-tuning eval (GLUE)... this may take a while.")
    run(
        f"bash scripts/eval_finetuning.sh --model_path {HUB_MODEL_REPO}",
        cwd=EVAL_REPO_DIR,
    )
    print("Fine-tuning eval complete.")

In [ ]:
# ── Cell 7b: Age of Acquisition eval — final model ──

MODEL_NAME      = HUB_MODEL_REPO.split("/")[-1]
aoa_result_path = Path(RESULTS_DIR) / MODEL_NAME / "main" / "aoa"

if aoa_result_path.exists() and any(aoa_result_path.iterdir()):
    print("AoA results already present — skipping.")
else:
    print("Running Age of Acquisition eval...")
    run(
        f"bash scripts/eval_aoa.sh {HUB_MODEL_REPO} {BACKEND} {TRACK}",
        cwd=EVAL_REPO_DIR,
    )
    print("AoA eval complete.")

In [ ]:
# ── Cell 7c: GlobalPIQA eval — main + all checkpoint revisions ──
# Covers both the final model and all chck_NM branches in one shot.

MODEL_NAME        = HUB_MODEL_REPO.split("/")[-1]
global_piqa_path  = Path(RESULTS_DIR) / MODEL_NAME / "main" / "zero_shot" / BACKEND / "global_piqa"

if global_piqa_path.exists() and any(global_piqa_path.iterdir()):
    print("GlobalPIQA results already present — skipping.")
else:
    print("Running GlobalPIQA eval (main + all checkpoint revisions)...")
    run(
        f"bash scripts/eval_zero_shot_global_piqa.sh {HUB_MODEL_REPO} {BACKEND} {TRACK}",
        cwd=EVAL_REPO_DIR,
    )
    print("GlobalPIQA eval complete.")

In [ ]:
# ── Cell 8: Collate predictions → submission JSON ──

print("Collating predictions...")
run(
    f"bash scripts/collate_preds.sh {HUB_MODEL_REPO} {BACKEND} {TRACK}",
    cwd=EVAL_REPO_DIR,
)

# Find the generated submission file
import json
submission_files = list(Path(RESULTS_DIR).rglob("*submission*.json")) + \
                   list(Path(RESULTS_DIR).rglob("*collated*.json"))
if submission_files:
    print(f"Submission file(s): {[str(f) for f in submission_files]}")
else:
    print("No submission JSON found — check results/ dir manually.")

print("Collation done.")

In [ ]:
# ── Cell 9: Print results summary ──

import json
from pathlib import Path

MODEL_NAME   = HUB_MODEL_REPO.split("/")[-1]
results_root = Path(RESULTS_DIR) / MODEL_NAME

# ── Load collated submission JSON ──
collated_path = results_root / f"all_full_preds_and_fast_scores_{BACKEND}.json"
collated = json.loads(collated_path.read_text()) if collated_path.exists() else {}
fast = collated.get("fast_eval_results", {})

# fast_eval_results.blimp is a list of 19 dicts (one per revision, FAST_REVISIONS order).
# Each dict maps subtask → accuracy (0–1). Average = mean BLiMP accuracy.
def _fast_avg(key: str, idx: int) -> float | None:
    entries = fast.get(key, [])
    if idx >= len(entries) or not isinstance(entries[idx], dict):
        return None
    vals = [v for v in entries[idx].values() if isinstance(v, (int, float))]
    return round(sum(vals) / len(vals), 4) if vals else None

# ── Reading scores from report.txt ──
def _reading_scores(rev: str):
    report = results_root / rev / "zero_shot" / BACKEND / "reading" / "report.txt"
    if not report.exists():
        return None, None
    et = spr = None
    for line in report.read_text().splitlines():
        if line.startswith("EYE TRACKING SCORE:"):
            et  = float(line.split(":")[1].strip())
        elif line.startswith("SELF-PACED READING SCORE:"):
            spr = float(line.split(":")[1].strip())
    return et, spr

print(f"Results summary for {HUB_MODEL_REPO}")
print("=" * 72)
print(f"\n{'Revision':<12}  {'BLiMP':>8}  {'BLiMP+Supp':>10}  {'Eye Track':>10}  {'SPR':>6}")
print("-" * 55)
for i, rev in enumerate(FAST_REVISIONS):
    blimp      = _fast_avg("blimp",            i)
    blimp_supp = _fast_avg("blimp_supplement", i)
    et, spr    = _reading_scores(rev)
    combined   = (
        round((blimp + blimp_supp) / 2, 4)
        if blimp is not None and blimp_supp is not None else None
    )
    b_s  = f"{blimp:.3f}"    if blimp    is not None else " n/a"
    bs_s = f"{combined:.3f}" if combined is not None else "  n/a"
    et_s = f"{et:.2f}"       if et       is not None else " n/a"
    sp_s = f"{spr:.2f}"      if spr      is not None else " n/a"
    print(f"  {rev:<12}  {b_s:>8}  {bs_s:>10}  {et_s:>10}  {sp_s:>6}")

# ── Full zero-shot — final model (main branch) reading scores ──
main_report = results_root / "main" / "zero_shot" / BACKEND / "reading" / "report.txt"
if main_report.exists():
    print("\nFull zero-shot reading (final model / main branch):")
    for line in main_report.read_text().splitlines():
        print(f"  {line}")

# ── Fine-tuning (predictions only — scored server-side by competition) ──
ft_dir = results_root / "main" / "finetune"
if ft_dir.exists():
    tasks = sorted(d.name for d in ft_dir.iterdir() if d.is_dir())
    print(f"\nFine-tuning predictions submitted for: {tasks}")

In [ ]:
# ── Cell 10: Push results to your HF results repo ──
# Results land at RESULTS_REPO/results/<model_name>/ so multiple models
# accumulate in one place without colliding.

from huggingface_hub import HfApi
from pathlib import Path

api = HfApi()

MODEL_NAME   = HUB_MODEL_REPO.split("/")[-1]
results_root = Path(RESULTS_DIR) / MODEL_NAME

if results_root.exists() and any(results_root.iterdir()):
    dest = f"results/{MODEL_NAME}"
    print(f"Pushing {results_root} → {RESULTS_REPO}/{dest}/ ...")
    api.upload_folder(
        folder_path=str(results_root),
        repo_id=RESULTS_REPO,
        path_in_repo=dest,
        repo_type="dataset",
        token=HF_TOKEN,
        commit_message=f"eval results: {MODEL_NAME}",
    )
    print("Results pushed.")

    # Also push any submission JSON under a submissions/<model_name>/ subfolder
    for jf in Path(RESULTS_DIR).glob("*.json"):
        api.upload_file(
            path_or_fileobj=str(jf),
            path_in_repo=f"submissions/{MODEL_NAME}/{jf.name}",
            repo_id=RESULTS_REPO,
            repo_type="dataset",
            token=HF_TOKEN,
            commit_message=f"submission: {MODEL_NAME}/{jf.name}",
        )
        print(f"Pushed submission file: submissions/{MODEL_NAME}/{jf.name}")
else:
    print(f"No results to push — {results_root} is empty or missing.")